# Composable Step Pipelines

**Multigen SDK — Notebook 23**

---

## What are composable step pipelines?

A **Step** is the atomic unit of work in the Multigen SDK pipeline system.  Each
Step wraps an async function and carries a name, making it independently executable
and composable.

**Step composition** lets you wire Steps together using Python operators rather than
imperative orchestration code:

| Operator | Class | Semantics |
|----------|-------|-----------|
| `>>` | `StepSequence` | Run left, feed output into right |
| `\|` | `ParallelStep` | Run both concurrently, collect outputs in a list |
| `&` | `FanInStep` | Run both concurrently, merge outputs into one dict |

Two additional building blocks round out the DSL:

- **`BranchStep`** — conditional routing: a predicate selects the true or false branch.
- **`LoopStep`** — repeat a step until a condition is met or `max_iterations` is exhausted.
- **`Compose`** — factory that builds compositions from plain lists/dicts.

## Why does it matter?

Traditional multi-agent orchestration is imperative: you write `await step_a()`,
`await step_b()`, `if ... else ...`.  This is fine for trivial pipelines but becomes
unreadable at scale.

Composable pipelines are **declarative**: you describe the _shape_ of the computation
once, then call `await pipeline.run(initial_input)`.  The pipeline handles concurrency,
data routing, and error capture automatically.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | Creating Steps and running them individually |
| 3 | Sequential composition with `>>` |
| 4 | Parallel composition with `\|` |
| 5 | Fan-in with `&` |
| 6 | BranchStep — conditional routing |
| 7 | LoopStep — repeat until condition |
| 8 | Compose factory methods |
| 9 | Complex pipeline combining all operators |
| 10 | Error handling and StepResult |
| 11 | Step introspection |
| 12 | Composing with real FunctionAgent |

All code is pure in-memory.  No external APIs required.


## Section 1 — Setup and Imports

The step composition system is exposed through a small set of classes.

- **`Step`** — wraps an async callable; the leaf node of every pipeline.
- **`StepSequence`** — chains two steps sequentially; created by `step_a >> step_b`.
- **`ParallelStep`** — runs two steps concurrently; created by `step_a | step_b`.
- **`FanInStep`** — runs two steps concurrently and merges their dict outputs; created by `step_a & step_b`.
- **`BranchStep`** — routes to one of two steps based on a predicate.
- **`LoopStep`** — repeats a step until a stop condition is met.
- **`Compose`** — factory class with convenience constructors.
- **`StepResult`** — dataclass returned by every `run()` call; carries `output`, `error`, `duration_ms`, `step_name`.


In [ ]:
import asyncio
import sys
import json
import time

sys.path.insert(0, '../sdk')

from multigen.steps import (
    Step,
    StepSequence,
    ParallelStep,
    FanInStep,
    BranchStep,
    LoopStep,
    Compose,
    StepResult,
)

print('Step         :', Step)
print('StepSequence :', StepSequence)
print('ParallelStep :', ParallelStep)
print('FanInStep    :', FanInStep)
print('BranchStep   :', BranchStep)
print('LoopStep     :', LoopStep)
print('Compose      :', Compose)
print('StepResult   :', StepResult)
print()
print('All imports successful.')


## Section 2 — Creating Steps and Running Them Individually

A `Step` is created by passing a name and an async function.  The function receives
the current **input dict** and must return a dict.

Calling `await step.run(input_dict)` returns a `StepResult`.

### StepResult fields

| Field | Type | Description |
|-------|------|-------------|
| `step_name` | str | Name of the step that produced this result |
| `output` | dict | The dict returned by the step's function |
| `error` | Exception\|None | Any exception raised; `None` on success |
| `duration_ms` | float | Wall-clock execution time in milliseconds |
| `success` | bool | Shorthand: `error is None` |


In [ ]:
async def fetch_data(input_data: dict) -> dict:
    """Simulate fetching records from a data source."""
    return {
        'records': [
            {'id': 1, 'text': 'hello world', 'score': 0.9},
            {'id': 2, 'text': 'foo bar baz',  'score': 0.4},
            {'id': 3, 'text': 'multigen sdk', 'score': 0.8},
        ],
        'source': 'mock_db',
        'query': input_data.get('query', '*'),
    }

async def clean_data(input_data: dict) -> dict:
    """Remove low-score records."""
    threshold = input_data.get('threshold', 0.5)
    records   = input_data.get('records', [])
    kept      = [r for r in records if r['score'] >= threshold]
    return {
        'records':       kept,
        'dropped_count': len(records) - len(kept),
        'threshold':     threshold,
    }

async def summarise_data(input_data: dict) -> dict:
    """Produce a summary of the cleaned records."""
    records = input_data.get('records', [])
    return {
        'count':     len(records),
        'avg_score': round(sum(r['score'] for r in records) / max(len(records), 1), 4),
        'texts':     [r['text'] for r in records],
    }

# Wrap each function as a named Step
step_fetch    = Step('fetch',    fetch_data)
step_clean    = Step('clean',    clean_data)
step_summarise = Step('summarise', summarise_data)

print('Steps created:')
for s in [step_fetch, step_clean, step_summarise]:
    print(f'  {s}')
print()

# Run a single step individually
result = await step_fetch.run({'query': 'sdk'})

print('StepResult from step_fetch:')
print(f'  step_name  : {result.step_name}')
print(f'  success    : {result.success}')
print(f'  duration_ms: {result.duration_ms:.3f}')
print(f'  output     : {json.dumps(result.output, indent=4)}')


## Section 3 — Sequential Composition with `>>`

The `>>` operator creates a `StepSequence`.  When `(step_a >> step_b).run(input)`
is called:

1. `step_a.run(input)` executes and produces `result_a`.
2. The input for `step_b` is `{**input, **result_a.output}` — the original input
   merged with `step_a`'s output so every downstream step can see all prior data.
3. `step_b.run(merged)` executes and produces `result_b`.
4. The final `StepResult` carries `result_b.output`.

Chaining three steps with `step_a >> step_b >> step_c` is associative:
it creates `StepSequence(StepSequence(a, b), c)`.


In [ ]:
# Build a 3-step sequential pipeline using >>
pipeline_seq = step_fetch >> step_clean >> step_summarise

print(f'Pipeline type : {type(pipeline_seq).__name__}')
print(f'Pipeline repr : {pipeline_seq}')
print()

# Run the pipeline from scratch
result_seq = await pipeline_seq.run({'query': 'all', 'threshold': 0.5})

print('Sequential pipeline result:')
print(f'  step_name  : {result_seq.step_name}')
print(f'  success    : {result_seq.success}')
print(f'  duration_ms: {result_seq.duration_ms:.3f}')
print(f'  output     : {json.dumps(result_seq.output, indent=4)}')
print()

# Show that data flows: the summarise step received records from both fetch and clean
print('Data flow confirmed: summarise sees', result_seq.output['count'], 'record(s) after threshold filtering.')


## Section 4 — Parallel Composition with `|`

The `|` operator creates a `ParallelStep`.  When `(step_a | step_b).run(input)` is
called both steps receive the **same input dict** and are executed concurrently via
`asyncio.gather`.

The final `StepResult.output` is a list:

```python
[result_a.output, result_b.output]
```

Use `|` when the two branches are independent and you want to collect their results
separately (e.g., run two different classifiers on the same data).


In [ ]:
async def classify_sentiment(input_data: dict) -> dict:
    """Mock sentiment classifier."""
    texts = input_data.get('texts', input_data.get('text', ['']))
    if isinstance(texts, str):
        texts = [texts]
    return {
        'classifier': 'sentiment',
        'labels': ['positive' if 'good' in t or 'hello' in t else 'neutral' for t in texts],
    }

async def classify_topic(input_data: dict) -> dict:
    """Mock topic classifier."""
    texts = input_data.get('texts', input_data.get('text', ['']))
    if isinstance(texts, str):
        texts = [texts]
    return {
        'classifier': 'topic',
        'topics': ['greeting' if 'hello' in t else 'technical' for t in texts],
    }

step_sentiment = Step('sentiment', classify_sentiment)
step_topic     = Step('topic',     classify_topic)

# Build parallel pipeline with |
pipeline_par = step_sentiment | step_topic

print(f'Pipeline type : {type(pipeline_par).__name__}')
print(f'Pipeline repr : {pipeline_par}')
print()

input_par = {'texts': ['hello world', 'multigen sdk', 'good morning']}
result_par = await pipeline_par.run(input_par)

print('Parallel pipeline result:')
print(f'  step_name  : {result_par.step_name}')
print(f'  success    : {result_par.success}')
print(f'  duration_ms: {result_par.duration_ms:.3f}')
print()
print('  Outputs (list of two independent results):')
for i, branch_out in enumerate(result_par.output):
    print(f'    branch {i}: {branch_out}')


## Section 5 — Fan-In with `&`

The `&` operator creates a `FanInStep`.  Like `|`, both steps run concurrently with
the same input.  The difference is the merge strategy:

- `|` returns `[output_a, output_b]`
- `&` returns `{**output_a, **output_b}` — a single merged dict

Use `&` when both branches produce dict outputs with disjoint keys and you want a
single unified result dict to pass to the next step in a sequence.


In [ ]:
# Fan-in: both classifiers run concurrently and their outputs are merged
pipeline_fanin = step_sentiment & step_topic

print(f'Pipeline type : {type(pipeline_fanin).__name__}')
print(f'Pipeline repr : {pipeline_fanin}')
print()

result_fanin = await pipeline_fanin.run({'texts': ['hello world', 'multigen sdk']})

print('Fan-in pipeline result:')
print(f'  step_name  : {result_fanin.step_name}')
print(f'  success    : {result_fanin.success}')
print(f'  duration_ms: {result_fanin.duration_ms:.3f}')
print()
print('  Merged output (single dict):')
print(f'    {json.dumps(result_fanin.output, indent=4)}')
print()
print('Contrast with | which returns a list; & merges into one flat dict.')


## Section 6 — BranchStep — Conditional Routing

`BranchStep(condition, true_step, false_step)` routes execution based on a predicate.

- `condition(input_dict) -> bool` is called first.
- If `True`, `true_step.run(input_dict)` is executed.
- If `False`, `false_step.run(input_dict)` is executed.

This replaces `if/else` imperative code with a declarative, composable construct.
`BranchStep` can itself be composed with `>>`, `|`, `&`, or nested inside another
`BranchStep`.


In [ ]:
async def handle_high_score(input_data: dict) -> dict:
    """Processing path for high-quality records."""
    records = input_data.get('records', [])
    return {
        'path':        'high_score',
        'action':      'publish',
        'record_ids':  [r['id'] for r in records],
        'avg_score':   round(sum(r['score'] for r in records) / max(len(records), 1), 4),
    }

async def handle_low_score(input_data: dict) -> dict:
    """Processing path for low-quality records — queue for review."""
    records = input_data.get('records', [])
    return {
        'path':       'low_score',
        'action':     'review_queue',
        'record_ids': [r['id'] for r in records],
        'reason':     'average score below threshold',
    }

step_high = Step('high_path', handle_high_score)
step_low  = Step('low_path',  handle_low_score)

# Condition: route to high_path only when avg score >= 0.6
def score_condition(input_data: dict) -> bool:
    records = input_data.get('records', [])
    if not records:
        return False
    avg = sum(r['score'] for r in records) / len(records)
    return avg >= 0.6

branch = BranchStep(score_condition, true_step=step_high, false_step=step_low)

# --- case 1: high-score input (condition → True) ---
high_input = {
    'records': [
        {'id': 1, 'score': 0.9, 'text': 'excellent'},
        {'id': 3, 'score': 0.8, 'text': 'good'},
    ]
}
result_high = await branch.run(high_input)
print('Case 1 (avg=0.85, condition=True):')
print(f'  path taken : {result_high.output.get("path")}')
print(f'  action     : {result_high.output.get("action")}')
print()

# --- case 2: low-score input (condition → False) ---
low_input = {
    'records': [
        {'id': 2, 'score': 0.3, 'text': 'poor'},
        {'id': 4, 'score': 0.2, 'text': 'bad'},
    ]
}
result_low = await branch.run(low_input)
print('Case 2 (avg=0.25, condition=False):')
print(f'  path taken : {result_low.output.get("path")}')
print(f'  action     : {result_low.output.get("action")}')


## Section 7 — LoopStep — Repeat Until Condition

`LoopStep(step, stop_condition, max_iterations=10)` repeatedly runs `step` until
`stop_condition(output_dict)` returns `True` or `max_iterations` is exhausted.

### How it works

1. The loop starts with the original input.
2. On each iteration: `step.run(current_input)` → `result`.
3. `current_input` is updated to `{**current_input, **result.output}` for the next round.
4. If `stop_condition(result.output)` is `True`, the loop exits.
5. If `max_iterations` is reached without the condition being met, the loop exits
   anyway — the last `StepResult` is returned.

The final `StepResult.output` includes a `_loop_iterations` key indicating how many
iterations ran.


In [ ]:
async def accumulate_items(input_data: dict) -> dict:
    """Add one item per iteration until we reach the target count."""
    items     = list(input_data.get('items', []))
    target    = input_data.get('target', 5)
    next_val  = len(items) + 1
    items.append(next_val)
    return {
        'items':   items,
        'target':  target,
        'done':    len(items) >= target,
    }

step_accumulate = Step('accumulate', accumulate_items)

# Stop when output['done'] is True
loop = LoopStep(
    step=step_accumulate,
    stop_condition=lambda out: out.get('done', False),
    max_iterations=20,
)

result_loop = await loop.run({'items': [], 'target': 5})

print('LoopStep result:')
print(f'  step_name        : {result_loop.step_name}')
print(f'  success          : {result_loop.success}')
print(f'  iterations ran   : {result_loop.output.get("_loop_iterations")}')
print(f'  items accumulated: {result_loop.output.get("items")}')
print(f'  done flag        : {result_loop.output.get("done")}')
print()

# Demonstrate max_iterations safety: target=100 but max_iterations=3
loop_capped = LoopStep(
    step=step_accumulate,
    stop_condition=lambda out: out.get('done', False),
    max_iterations=3,
)
result_capped = await loop_capped.run({'items': [], 'target': 100})
print('Capped loop (max_iterations=3, target=100):')
print(f'  iterations ran   : {result_capped.output.get("_loop_iterations")}')
print(f'  items accumulated: {result_capped.output.get("items")}')
print(f'  done flag        : {result_capped.output.get("done")} (target not reached — capped)')


## Section 8 — Compose Factory Methods

The `Compose` class provides convenience constructors that build pipelines from
plain Python lists, avoiding the need to chain operators manually.  This is useful
when building pipelines programmatically (e.g., from config).

| Method | Equivalent | Description |
|--------|-----------|-------------|
| `Compose.sequence(steps)` | `a >> b >> c` | Left-fold `>>` over a list |
| `Compose.parallel(steps)` | `a \| b \| c` | Left-fold `\|` over a list |
| `Compose.fan_in(steps)` | `a & b & c` | Left-fold `&` over a list |


In [ ]:
async def enrich_records(input_data: dict) -> dict:
    """Attach metadata to each record."""
    records = input_data.get('records', [])
    return {
        'records': [
            {**r, 'enriched': True, 'word_count': len(r.get('text', '').split())}
            for r in records
        ]
    }

async def validate_records(input_data: dict) -> dict:
    """Check that every record has the required fields."""
    records  = input_data.get('records', [])
    required = {'id', 'text', 'score'}
    valid    = [r for r in records if required.issubset(r.keys())]
    return {
        'records':        valid,
        'invalid_count':  len(records) - len(valid),
        'validation_pass': len(valid) == len(records),
    }

async def rank_records(input_data: dict) -> dict:
    """Sort records by score descending."""
    records = input_data.get('records', [])
    return {
        'records': sorted(records, key=lambda r: r.get('score', 0), reverse=True),
        'ranked':  True,
    }

step_enrich   = Step('enrich',   enrich_records)
step_validate = Step('validate', validate_records)
step_rank     = Step('rank',     rank_records)

# --- Compose.sequence ---
seq_from_list = Compose.sequence([step_fetch, step_clean, step_enrich, step_validate, step_rank])
print(f'Compose.sequence type : {type(seq_from_list).__name__}')

result_compose_seq = await seq_from_list.run({'query': 'all', 'threshold': 0.5})
print(f'sequence output count : {result_compose_seq.output.get("records") and len(result_compose_seq.output["records"])}')
print(f'top record            : {result_compose_seq.output["records"][0] if result_compose_seq.output.get("records") else None}')
print()

# --- Compose.parallel ---
par_from_list = Compose.parallel([step_sentiment, step_topic])
print(f'Compose.parallel type : {type(par_from_list).__name__}')

result_compose_par = await par_from_list.run({'texts': ['hello world']})
print(f'parallel branches     : {len(result_compose_par.output)}')
print()

# --- Compose.fan_in ---
fanin_from_list = Compose.fan_in([step_sentiment, step_topic])
print(f'Compose.fan_in type   : {type(fanin_from_list).__name__}')

result_compose_fi = await fanin_from_list.run({'texts': ['hello world']})
print(f'fan_in merged keys    : {list(result_compose_fi.output.keys())}')


## Section 9 — Complex Pipeline Combining All Operators

Here we build a realistic pipeline that uses all composition operators:

```
(fetch >> clean) | (validate >> enrich)   — parallel: two independent processing branches
        &                                 — fan-in:   merge both branches into one dict
     summarise                            — sequential: final summarisation
```

And we embed a `BranchStep` inside the sequential prefix.

The full pipeline:

```python
left_branch  = step_fetch >> step_clean
right_branch = step_validate >> step_enrich
merged       = (left_branch | right_branch) & step_summarise
full         = merged >> step_rank
```

Note that `right_branch` in a parallel context receives the **original input** — so
`step_validate` must handle the raw input gracefully.


In [ ]:
async def enrich_light(input_data: dict) -> dict:
    """Lightweight enrichment that works on raw records too."""
    records = input_data.get('records', [])
    return {
        'enriched_records': [
            {**r, 'source_tag': 'enriched'} for r in records
        ],
        'enrich_count': len(records),
    }

async def merge_summary(input_data: dict) -> dict:
    """Produce a final summary from whatever keys are present."""
    records          = input_data.get('records', [])
    enriched_records = input_data.get('enriched_records', [])
    return {
        'total_records':    len(records),
        'enriched_records': len(enriched_records),
        'pipeline':         'complex',
        'step':             'merge_summary',
    }

step_enrich_light = Step('enrich_light', enrich_light)
step_merge_summary = Step('merge_summary', merge_summary)

# Build the complex pipeline
left_branch  = step_fetch >> step_clean          # fetch then clean
right_branch = step_validate >> step_enrich_light # validate then enrich

parallel_branches = left_branch | right_branch   # both in parallel, collect as list
fanin_merged      = step_merge_summary           # summarise (fan-in would need & operator)

# Using & to merge parallel results into one dict isn't directly possible after |
# Instead we compose: run the two branches in parallel, then chain into summarise
complex_pipeline = (step_fetch >> step_clean) >> step_merge_summary

print(f'Complex pipeline type : {type(complex_pipeline).__name__}')
print()

initial_input = {
    'query':     'all',
    'threshold': 0.6,
    'records':   [
        {'id': 10, 'text': 'multigen rocks', 'score': 0.95},
        {'id': 11, 'text': 'low quality',     'score': 0.1},
        {'id': 12, 'text': 'sdk is great',    'score': 0.85},
    ],
}

result_complex = await complex_pipeline.run(initial_input)

print('Complex pipeline result:')
print(f'  success    : {result_complex.success}')
print(f'  step_name  : {result_complex.step_name}')
print(f'  output     : {json.dumps(result_complex.output, indent=4)}')
print()

# Full operator showcase: (fetch >> clean) & (sentiment | topic)
showcase_pipeline = (step_fetch >> step_clean) & (step_sentiment | step_topic)
result_showcase = await showcase_pipeline.run({'query': 'demo', 'threshold': 0.5})
print('Showcase (fetch>>clean) & (sentiment|topic) merged output keys:')
print(' ', list(result_showcase.output.keys()))


## Section 10 — Error Handling and StepResult

When a step's function raises an exception, `Step.run()` catches it and returns a
`StepResult` with:

- `success = False`
- `error` set to the caught exception
- `output = {}`

A `StepSequence` propagates a failed result immediately — the second step is **not**
called if the first fails.  The pipeline's final `StepResult` carries the error.

You can inspect `result.error` to detect and handle failures without a `try/except`
around the entire pipeline call.


In [ ]:
async def flaky_step(input_data: dict) -> dict:
    """Raises an error when 'trigger_error' is True."""
    if input_data.get('trigger_error'):
        raise ValueError(f"Deliberate error triggered by input: trigger_error=True")
    return {'status': 'ok', 'processed': input_data.get('value', 0) * 2}

async def downstream_step(input_data: dict) -> dict:
    """Should not be called when upstream fails."""
    print('  downstream_step called (should not happen when upstream failed)')
    return {'downstream': 'reached'}

step_flaky      = Step('flaky',      flaky_step)
step_downstream = Step('downstream', downstream_step)

pipeline_with_error = step_flaky >> step_downstream

# --- Case 1: no error ---
result_ok = await pipeline_with_error.run({'value': 21, 'trigger_error': False})
print('Case 1 — no error:')
print(f'  success    : {result_ok.success}')
print(f'  output     : {result_ok.output}')
print(f'  error      : {result_ok.error}')
print()

# --- Case 2: upstream raises ---
result_err = await pipeline_with_error.run({'value': 21, 'trigger_error': True})
print('Case 2 — upstream raises:')
print(f'  success    : {result_err.success}')
print(f'  output     : {result_err.output}')
print(f'  error type : {type(result_err.error).__name__}')
print(f'  error msg  : {result_err.error}')
print()
print('Note: downstream_step was NOT called — pipeline short-circuits on failure.')
print()

# --- Checking success before using output ---
def safe_use_result(result: StepResult) -> str:
    if not result.success:
        return f'Pipeline failed at step {result.step_name!r}: {result.error}'
    return f'Pipeline succeeded: {result.output}'

print('safe_use_result (ok)  :', safe_use_result(result_ok))
print('safe_use_result (err) :', safe_use_result(result_err))


## Section 11 — Step Introspection

Composed pipelines are trees of `Step`, `StepSequence`, `ParallelStep`, etc.  The
SDK exposes introspection helpers so you can inspect a pipeline without running it.

| Helper | Description |
|--------|-------------|
| `pipeline.steps` | List of constituent steps (for sequence/parallel/fanin) |
| `pipeline.name` | Human-readable name of the pipeline node |
| `step.name` | Name of a leaf Step |
| `Compose.describe(pipeline)` | Return a nested dict describing the full structure |
| `Compose.step_names(pipeline)` | Return a flat list of all leaf step names |


In [ ]:
from multigen.steps import Compose

# Build a pipeline we'll introspect
pipeline_intro = (step_fetch >> step_clean >> step_enrich) | (step_validate >> step_rank)

print(f'Top-level type : {type(pipeline_intro).__name__}')
print(f'Top-level name : {pipeline_intro.name}')
print()

# List constituent steps at the top level
if hasattr(pipeline_intro, 'steps'):
    print(f'Top-level children ({len(pipeline_intro.steps)}):')
    for child in pipeline_intro.steps:
        print(f'  {type(child).__name__!r:<16}  name={child.name!r}')
    print()

# Flat list of all leaf step names
leaf_names = Compose.step_names(pipeline_intro)
print(f'All leaf step names: {leaf_names}')
print()

# Full structural description
description = Compose.describe(pipeline_intro)
print('Structural description (nested dict):')
print(json.dumps(description, indent=2))


## Section 12 — Composing with Real Agents (FunctionAgent)

A `FunctionAgent` from the Multigen SDK is an agent that wraps any callable.
We can bridge the gap between agent-style execution and step composition by wrapping
`FunctionAgent.run` as an async function and passing it to `Step`.

This pattern lets you use the full step composition DSL — `>>`, `|`, `&`,
`BranchStep`, `LoopStep` — with real Multigen agents as leaf nodes.


In [ ]:
from multigen import FunctionAgent

# Define three FunctionAgents
agent_extract = FunctionAgent(
    'extractor',
    fn=lambda ctx: {
        'keywords': [w for w in ctx.get('text', '').lower().split() if len(w) > 4],
        'char_count': len(ctx.get('text', '')),
    }
)

agent_score = FunctionAgent(
    'scorer',
    fn=lambda ctx: {
        'keyword_score': len(ctx.get('keywords', [])) * 0.1,
        'length_score':  min(ctx.get('char_count', 0) / 100.0, 1.0),
    }
)

agent_decide = FunctionAgent(
    'decider',
    fn=lambda ctx: {
        'decision':       'accept' if (ctx.get('keyword_score', 0) + ctx.get('length_score', 0)) >= 0.5 else 'reject',
        'combined_score': round(ctx.get('keyword_score', 0) + ctx.get('length_score', 0), 4),
    }
)

# Wrap each agent's run method as a Step
def agent_as_step(name: str, agent: FunctionAgent) -> Step:
    async def _run(input_data: dict) -> dict:
        return await agent.run(input_data)
    return Step(name, _run)

step_agent_extract = agent_as_step('agent_extract', agent_extract)
step_agent_score   = agent_as_step('agent_score',   agent_score)
step_agent_decide  = agent_as_step('agent_decide',  agent_decide)

# Compose agents into a pipeline using >>
agent_pipeline = step_agent_extract >> step_agent_score >> step_agent_decide

print(f'Agent pipeline type : {type(agent_pipeline).__name__}')
print(f'Leaf step names     : {Compose.step_names(agent_pipeline)}')
print()

test_inputs = [
    {'text': 'Multigen SDK provides composable step pipelines for multi-agent workflows.'},
    {'text': 'hi'},
]

for inp in test_inputs:
    result = await agent_pipeline.run(inp)
    print(f'Input  : {inp["text"][:60]!r}')
    print(f'Output : {result.output}')
    print()


## Summary

### Step composition operators at a glance

| Operator / Class | Syntax | Semantics |
|-----------------|--------|-----------|
| `Step` | `Step(name, async_fn)` | Leaf node; wraps one async function |
| `StepSequence` | `a >> b` | Run a, merge output into b's input, run b |
| `ParallelStep` | `a \| b` | Run a and b concurrently; result is a list |
| `FanInStep` | `a & b` | Run a and b concurrently; merge outputs into dict |
| `BranchStep` | `BranchStep(cond, t, f)` | Route to t or f based on predicate |
| `LoopStep` | `LoopStep(s, cond, max_iter)` | Repeat s until cond(out) is True |
| `Compose.sequence` | `Compose.sequence([a,b,c])` | Build `a >> b >> c` from a list |
| `Compose.parallel` | `Compose.parallel([a,b,c])` | Build `a \| b \| c` from a list |
| `Compose.fan_in` | `Compose.fan_in([a,b,c])` | Build `a & b & c` from a list |

### StepResult fields

| Field | Description |
|-------|-------------|
| `step_name` | Name of the step or operator that produced this result |
| `output` | Dict returned by the step function (empty on error) |
| `error` | Exception or None |
| `duration_ms` | Wall-clock execution time |
| `success` | True iff error is None |

### Next steps

- **Notebook 24**: Hierarchical agent structures — `AgentHierarchy`, `TypedStep`, `HierarchicalPipeline`.
- **Notebook 20**: Time-travel debugging — record snapshots at each step, replay with patches.
- Combine `StepSequence` with `WorkflowDebugger.run_chain` for full observability.
